# Romanian dataset analysis

## Load

In [2]:
import json
from collections import Counter
import numpy as np

with open('../datasets/ro-gold/ro_gold_final.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f'loaded {len(dataset)} entries')


loaded 1952 entries


## Fleiss kappa (polarity)

In [3]:
CATS = ['positive', 'neutral', 'negative']
ORDER = {'positive': 0, 'neutral': 1, 'negative': 2}

def build_matrix(records):
    rows = []
    for r in records:
        pols = [a['polarity'] for a in r['annotators'] if a['polarity'] is not None]
        if len(pols) != 3:
            continue
        rows.append([pols.count(c) for c in CATS])
    return np.array(rows, dtype=float)

def fleiss_kappa(M):
    N, k = M.shape
    n = M.sum(axis=1)[0]
    P_i = ((M ** 2).sum(axis=1) - n) / (n * (n - 1))
    P_bar = P_i.mean()
    p_j = M.sum(axis=0) / (N * n)
    P_e = (p_j ** 2).sum()
    return (P_bar - P_e) / (1 - P_e), P_bar, P_e

def fleiss_kappa_weighted(M, W):
    N, k = M.shape
    n = M.sum(axis=1)[0]
    P_i = np.zeros(N)
    for idx in range(N):
        row = M[idx]
        s = 0.0
        for a in range(k):
            for b in range(k):
                s += W[a, b] * (row[a] * (row[a] - 1) if a == b else row[a] * row[b])
        P_i[idx] = s / (n * (n - 1))
    P_bar = P_i.mean()
    p_j = M.sum(axis=0) / (N * n)
    P_e = sum(W[a, b] * p_j[a] * p_j[b] for a in range(k) for b in range(k))
    return (P_bar - P_e) / (1 - P_e)

M = build_matrix(dataset)
print(f'items with full 3-rater polarity coverage: {M.shape[0]}')

k_un, Pbar, Pe = fleiss_kappa(M)
print(f'fleiss kappa (unweighted):         {k_un:.4f}   [P_bar={Pbar:.4f}, P_e={Pe:.4f}]')

W_lin = np.array([[1 - abs(ORDER[a] - ORDER[b]) / 2 for b in CATS] for a in CATS])
W_quad = np.array([[1 - (ORDER[a] - ORDER[b]) ** 2 / 4 for b in CATS] for a in CATS])
print(f'fleiss kappa (linear-weighted):    {fleiss_kappa_weighted(M, W_lin):.4f}')
print(f'fleiss kappa (quadratic-weighted): {fleiss_kappa_weighted(M, W_quad):.4f}')


items with full 3-rater polarity coverage: 1924
fleiss kappa (unweighted):         0.4578   [P_bar=0.8091, P_e=0.6479]
fleiss kappa (linear-weighted):    0.4756
fleiss kappa (quadratic-weighted): 0.5060


## Label distribution

In [4]:
pol_c = Counter(r['polarity'] for r in dataset)
total = len(dataset)
print(f'final polarity distribution ({total} entries):')
for lab in ['positive', 'neutral', 'negative']:
    n = pol_c.get(lab, 0)
    print(f'  {lab:10s} {n:5d} ({n/total:.2%})')
unresolved = pol_c.get(None, 0)
if unresolved:
    print(f'  unresolved {unresolved:5d} ({unresolved/total:.2%})')


final polarity distribution (1952 entries):
  positive     135 (6.92%)
  neutral     1612 (82.58%)
  negative     205 (10.50%)


In [5]:
emo_c = Counter(a['primary_emotion'] for r in dataset for a in r['annotators'] if a['primary_emotion'])
total_emo = sum(emo_c.values())
print(f'primary emotion distribution (rater-level, {total_emo} votes):')
for emo, n in emo_c.most_common():
    print(f'  {emo:10s} {n:5d} ({n/total_emo:.2%})')


primary emotion distribution (rater-level, 5828 votes):
  none        4604 (79.00%)
  joy          390 (6.69%)
  anger        329 (5.65%)
  fear         256 (4.39%)
  sadness      188 (3.23%)
  love          61 (1.05%)


## Per-comment-type breakdown

In [6]:
from collections import defaultdict

type_pol = defaultdict(Counter)
for r in dataset:
    type_pol[r['comment_type']][r['polarity']] += 1

types = sorted(type_pol, key=lambda t: -sum(type_pol[t].values()))
total = len(dataset)

print(f'{"type":<16} {"n":>5} {"share":>7}  {"pos":>7} {"neu":>7} {"neg":>7}')
print('-' * 56)
for t in types:
    counts = type_pol[t]
    n = sum(counts.values())
    pos = counts.get('positive', 0)
    neu = counts.get('neutral', 0)
    neg = counts.get('negative', 0)
    print(f'{t:<16} {n:>5} {n/total:>6.1%}  '
          f'{pos/n:>6.1%} {neu/n:>6.1%} {neg/n:>6.1%}')

print('-' * 56)
pos_all = sum(c.get('positive', 0) for c in type_pol.values())
neu_all = sum(c.get('neutral', 0) for c in type_pol.values())
neg_all = sum(c.get('negative', 0) for c in type_pol.values())
print(f'{"overall":<16} {total:>5} {1.0:>6.1%}  '
      f'{pos_all/total:>6.1%} {neu_all/total:>6.1%} {neg_all/total:>6.1%}')


type                 n   share      pos     neu     neg
--------------------------------------------------------
review_comment    1375  70.4%    2.6%  88.5%   8.9%
issue_comment      384  19.7%   11.7%  74.5%  13.8%
review             193   9.9%   28.0%  56.5%  15.5%
--------------------------------------------------------
overall           1952 100.0%    6.9%  82.6%  10.5%
